# Fine-Tune Mistral-7B-v0.3 with LoRA on Docker Q&A

The fine-tuning notebook for **every** LoRA run in the project — r=16 (Task #5) and the r=8 / r=32 rank ablations (Task #7). We load the **base** Mistral-7B in 4-bit, attach LoRA adapters, train 3 epochs on `data/train/train_1k.jsonl`, and save the adapter so the eval notebook can score it.

**To pick which run:** set `CONFIG` + `RUN_NAME` in the **Run Parameters** cell (section 2.5) below, then run top-to-bottom. Everything downstream keys off those two variables.

**Runtime budget**: ~1–2 hours on a free Colab T4.

**Before you run this notebook, make sure:**
1. Runtime → Change runtime type → **T4 GPU** is selected.
2. You've accepted Mistral-7B's license on HuggingFace (it's gated).
3. `HF_TOKEN` is set in Colab Secrets (🔑 left sidebar). `WANDB_API_KEY` is optional.
4. The latest commits are pushed to the GitHub repo cloned below — Colab pulls a fresh copy each run (so the ablation configs must be pushed first).

## 1. Confirm GPU is available

Should print a T4 with ~15 GB VRAM. If `nvidia-smi` fails or shows no GPU, switch the runtime type and restart.

In [ ]:
!nvidia-smi

## 2. Clone the project

Replace `<your-username>/LLM_Finetune` with your actual repo. For a private repo, use `https://<token>@github.com/<you>/LLM_Finetune.git` with a fine-grained PAT that has read access.

In [ ]:
!git clone https://github.com/Gabriel-Kevorkian/LLM_Finetune.git
%cd LLM_Finetune

## 2.5 Run parameters — **EDIT THESE FOR EACH RUN**

This is the only cell you change between runs. `RUN_NAME` must match the `run_name` field inside the chosen `CONFIG` yaml. Every cell below reads these two variables — the train command, the adapter inspection, the smoke test, and the Drive verification all interpolate `{CONFIG}` / `{RUN_NAME}`, so there are no other hardcoded paths to forget.

In [ ]:
# === RUN PARAMETERS — change these TWO lines for each run, then run top-to-bottom ===
#   r=16 (default, done Saturday):  configs/base.yaml            + RUN_NAME="r16"
#   r=8  ablation (Task #7):        configs/ablation_rank_8.yaml  + RUN_NAME="r8"
#   r=32 ablation (Task #7):        configs/ablation_rank_32.yaml + RUN_NAME="r32"
# RUN_NAME MUST match the run_name field inside the chosen CONFIG yaml — it names
# the output folders models/adapters/<RUN_NAME>/ and results/runs/<RUN_NAME>/.
CONFIG   = "configs/ablation_rank_8.yaml"
RUN_NAME = "r8"
print(f"CONFIG   = {CONFIG}")
print(f"RUN_NAME = {RUN_NAME}")

## 3. Install training dependencies

We need **`unsloth`** (fast LoRA kernels — it pulls compatible `trl`/`peft`/`accelerate`/`bitsandbytes` as transitive deps) plus our `requirements.txt`.

**Why we do NOT `pip install -U trl peft accelerate bitsandbytes`**: Unsloth's compiled cache is generated against a *specific* TRL version. Upgrading TRL afterwards breaks Unsloth's monkey-patches with errors like `TypeError: SFTConfig.__init__() got an unexpected keyword argument 'push_to_hub_token'`. Let Unsloth pin its own deps.

Install takes ~2 minutes.

In [ ]:
%%capture
# NO `-U` on trl/peft/accelerate/bitsandbytes — that would break Unsloth's
# compiled cache. unsloth's own setup.py is the source of truth for pins.
!pip install -q unsloth datasets
!pip install -q -r requirements.txt

## 4. Load secrets

We read tokens from the Colab Secrets panel into env vars. `HF_TOKEN` is required to download the gated Mistral weights. `WANDB_API_KEY` is optional — if present, we enable Weights & Biases logging on the training run.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]               = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    USE_WANDB = True
except Exception:
    USE_WANDB = False

print(f"HF_TOKEN:       {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"WANDB_API_KEY:  {'set' if USE_WANDB else 'not set (W&B disabled)'}")

## 5. Mount Google Drive and redirect output dirs to Drive

**This step is what makes the run survivable.** Colab's `/content/` filesystem is wiped on disconnect. We mount Drive, then **symlink** `models/adapters/` and `results/runs/` to paths inside `/content/drive/MyDrive/LLM_Finetune/`, so every write lands on Drive automatically and survives a runtime restart.

You'll get a Drive auth popup the first time — click through and allow all permissions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, pathlib

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/LLM_Finetune")
(DRIVE_ROOT / "models/adapters").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "results/runs").mkdir(parents=True, exist_ok=True)

# Replace local output dirs with symlinks pointing INTO Drive. From now on any
# write to models/adapters/... or results/runs/... lands on Drive automatically.
# We always recreate the symlinks so re-running this cell is safe.
for local_path_str, drive_path in [
    ("/content/LLM_Finetune/models/adapters", DRIVE_ROOT / "models/adapters"),
    ("/content/LLM_Finetune/results/runs",    DRIVE_ROOT / "results/runs"),
]:
    local_path = pathlib.Path(local_path_str)
    if local_path.is_symlink():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(drive_path, local_path)
    print(f"  {local_path} -> {drive_path}")

print("\nDrive mounted and output dirs symlinked — writes survive runtime restarts.")

## 6. Run the fine-tune

Trains the LoRA adapter defined by `{CONFIG}` (the cell reads the variable you set in section 2.5) and saves it to `models/adapters/<RUN_NAME>/` → symlinked to Drive. A training summary lands in `results/runs/<RUN_NAME>/`.

**Loss expectations**: average loss usually ends ~1.0–1.4; what matters is step-level loss dropping over time (~1.9 early → ~0.95 by step 375). **Runtime**: ~55–60 min on a T4. Don't close the tab.

In [ ]:
# Defensive cleanup: Unsloth caches a compiled SFTTrainer wrapper to
# unsloth_compiled_cache/ on first run. A stale cache (compiled against a
# different TRL version) makes Unsloth crash on reuse. Nuke it to force a
# fresh compile against the currently-installed TRL.
!rm -rf unsloth_compiled_cache

wandb_flag = "--wandb" if USE_WANDB else ""
!python scripts/04_train.py --config {CONFIG} {wandb_flag}

## 7. Inspect the saved adapter

Two output locations (both symlinked to Drive): `models/adapters/<RUN_NAME>/` (the LoRA weights + tokenizer — what the eval notebook loads) and `results/runs/<RUN_NAME>/` (frozen `config.yaml` + `train_summary.json`). **Confirm the `run_name` in the summary matches what you intended** — that's the check that catches a wrong-config run.

In [ ]:
!ls -lh models/adapters/{RUN_NAME}/
print()
!cat results/runs/{RUN_NAME}/train_summary.json

## 8. Smoke test — generate one answer from the fine-tuned model

Quick visual check that the adapter loaded and produces Docker-flavoured answers. **This is NOT the formal eval** (that's the eval notebook, scripts/05) — just a sanity check.

In [ ]:
import sys
sys.path.insert(0, ".")

!ls -la models/adapters/{RUN_NAME}/

from unsloth import FastLanguageModel
from peft import PeftModel
from src.data.format_prompts import format_chat, ensure_chat_template

# Inference is a TWO-STEP load:
#   1) Load the same 4-bit base model we trained from.
#   2) Layer our trained LoRA adapter ON TOP via PeftModel.from_pretrained.
# models/adapters/<RUN_NAME>/ holds only the LoRA delta (adapter_config.json +
# adapter_model.safetensors) — NOT a full model. A single
# FastLanguageModel.from_pretrained(adapter_path) would fail ("No config file found").
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
ensure_chat_template(tokenizer)

model = PeftModel.from_pretrained(model, f"models/adapters/{RUN_NAME}")
FastLanguageModel.for_inference(model)

question = "What is the difference between COPY and ADD in a Dockerfile?"
prompt   = format_chat(question, tokenizer)
inputs   = tokenizer(prompt, return_tensors="pt").to("cuda")

out_ids  = model.generate(**inputs, max_new_tokens=256, do_sample=False)
response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Q:", question)
print("A:", response)

## 9. Verify the adapter is on Drive

Because section 5 symlinked the output dirs to Drive, the adapter is already persisted. This cell just verifies it landed under the right `RUN_NAME` folder. After it passes, you can close the runtime; the eval notebook reads from this same Drive path.

In [ ]:
import os

DRIVE_ADAPTER = f"/content/drive/MyDrive/LLM_Finetune/models/adapters/{RUN_NAME}"
DRIVE_RESULTS = f"/content/drive/MyDrive/LLM_Finetune/results/runs/{RUN_NAME}"

print(f"adapter on Drive?  {os.path.isdir(DRIVE_ADAPTER)}")
print(f"results on Drive?  {os.path.isdir(DRIVE_RESULTS)}")

if os.path.isdir(DRIVE_ADAPTER):
    print(f"\nAdapter files at {DRIVE_ADAPTER}:")
    !ls -lh {DRIVE_ADAPTER}
    print(f"\nResults at {DRIVE_RESULTS}:")
    !ls -lh {DRIVE_RESULTS}
    print()
    !cat {DRIVE_RESULTS}/train_summary.json
else:
    print("\n[WARN] Adapter not on Drive. Check that section 5 ran successfully.")
    print("If you skipped section 5, run Option B below to grab a local zip backup.")

In [ ]:
# Option B (backup): zip + download to local machine.
# Uncomment if you didn't mount Drive in section 5 and want a local copy instead.
#
# import shutil
# shutil.make_archive(f"{RUN_NAME}_adapter", "zip", f"models/adapters/{RUN_NAME}")
# from google.colab import files
# files.download(f"{RUN_NAME}_adapter.zip")

## 10. Commit the small artifacts back to git

The adapter itself is gitignored (`models/` excluded). But the frozen `config.yaml` + `train_summary.json` in `results/runs/<RUN_NAME>/` are tiny and reproducible — those go to git for the audit trail.

**You git-push from your local machine** (to keep history free of `colab` author commits): copy `results/runs/<RUN_NAME>/config.yaml` + `train_summary.json` from your Drive folder into your local repo, then `git add` + `git commit` + `git push`.